# Day 13 — Delta Lake Change Data Feed (CDF) & Incremental Processing

- if CDF NOT ENABLED, then you have to **compare the full table** to identify the changes
- if CDF ENABLED --> Delta Lake exposes the **row-level changes** associated with table commits.

CDF allows downstream consumers to** process changes instead of repeatedly comparing entire table** snapshots.

## CDF using SQL
```
    ALTER TABLE workspace.customer360_bronze.customers_cdf_day13
    SET TBLPROPERTIES (
        delta.enableChangeDataFeed = true
    );


    SELECT
    custId,
    customerName,
    status,
    balance,
    _change_type,
    _commit_version,
    _commit_timestamp
FROM table_changes(
    'workspace.customer360_bronze.customers_cdf_day13',
    2
)
ORDER BY _commit_version, custId;
````


## CDF using pyspark

````
    source_table = "workspace.customer360_bronze.customers_cdf_day13"

    dfCDFStream = (
        spark.readStream
        .format("delta")
        .option("readChangeFeed", "true")
        .option("startingVersion", 2)
        .table(source_table)
    )

    target_table = "workspace.customer360_silver.customers_cdf_history_day13"

checkpoint_path = (
    "/Volumes/workspace/customer360_bronze/"
    "ingestion_metadata/checkpoints/"
    "customers_cdf_history_day13"
)

query = (
    dfCDFStream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()
````

## Interview question
## Suppose you run the streaming query three times without adding new source records. 
## Will the Silver history table contain 5, 10, or 15 records? Why?
CDF identifies changed rows; checkpointing prevents reprocessing previously committed source progress during normal recovery. With a Delta sink, this supports exactly-once processing under the supported guarantees.

Important interview concept
Why do we filter out update_preimage?

Because update_preimage contains the old values before an UPDATE. For an SCD Type 1 current-state table, we want the latest values from update_postimage.

Also, notice that our MERGE handles three scenarios:
- MATCHED + DELETE: Removes an existing customer.
- MATCHED + INSERT/UPDATE: Updates an existing customer.
- NOT MATCHED + INSERT/UPDATE: Inserts a new customer.
A DELETE for a customer that does not exist in the target is ignored.


````

%sql
dfLatestChanges.createOrReplaceTempView("latest_customer_changes")


MERGE INTO workspace.customer360_silver.customers_current_day13 AS target
USING latest_customer_changes AS source
ON target.custId = source.custId

WHEN MATCHED AND source._change_type = 'delete'
THEN DELETE

WHEN MATCHED AND source._change_type IN ('insert', 'update_postimage')
THEN UPDATE SET
    target.customerName = source.customerName,
    target.status = source.status,
    target.balance = source.balance

WHEN NOT MATCHED AND source._change_type IN ('insert', 'update_postimage')
THEN INSERT (
    custId,
    customerName,
    status,
    balance
)
VALUES (
    source.custId,
    source.customerName,
    source.status,
    source.balance
);
````

%md
**Q1. What is the difference between Delta Change Data Feed and CDC?**

CDF is a Delta Lake feature that exposes row-level changes made to a Delta table. CDC is a broader approach for capturing and propagating changes between systems. CDF can serve as a source for a CDC pipeline.

**Q2. How would you build an incremental pipeline using CDF?**

I would enable CDF on the source Delta table, read changes using Structured Streaming, and maintain a checkpoint to track processing progress. For a current-state Silver table, I would apply inserts, updates, and deletes using Delta MERGE, with appropriate change ordering and idempotent retry handling.

**Q3. What is the difference between CDF and SCD Type 2?**

CDF provides a feed of changes from a Delta table. SCD Type 2 is a data-modeling technique that preserves historical versions of business records. CDF can be used to drive an SCD Type 2 pipeline, but it does not automatically create SCD Type 2 history.